# 13 · Memory on the CLI boundary + per-term model  (~2–3 h)

> Part of the v0.2.0 experiments (notebooks 10–19), run with the fastapriori 0.2.0 build and the `bench/exp_runners.py` harness. Defaults: 1 warm-up + 3 measured runs per cell, full repetitions even for slow cells, explicit thread counts, failures recorded. Never run two of these notebooks at the same time.

Feeds §6.5 (Fig. 4) and the memory model in §3.3.

* `hybcli_*` run the Rust kernel in a numpy-only process (no pandas, no fastapriori import),
  reading pre-encoded arrays: the same boundary as the Borgelt/LCM command-line runs.
* `hybcli_load` is the same process without the call: interpreter + input baseline.
* Report both raw peak RSS and peak − load-only.
* The per-term model (index, anchor TID cache, thread buffers) is computed below from
  the data and the mined (k−1) level.

Output: `results/runs/<campaign>/memory_boundary.csv`, `results/runs/<campaign>/memory_terms.csv`.

`FA_MODE=replay` analyses the saved campaign (`vm_20261029` unless `FA_CAMPAIGN` says otherwise): no mining and no data needed; tables and figures go to `results/derived/<campaign>/`.


In [ ]:
import os, sys, json, math, time
from pathlib import Path
BUNDLE = Path.cwd().resolve().parent     # notebooks run from <bundle>/notebooks
assert (BUNDLE / "bench" / "exp_runners.py").exists(), "start Jupyter in <bundle>/notebooks"
sys.path.insert(0, str(BUNDLE / "bench"))
# Results of these notebooks go to results/runs/<FA_CAMPAIGN>/ (set it before starting Jupyter
# to keep campaigns apart; every run_grid call also appends to manifest.jsonl there).
REPLAY = os.environ.get("FA_MODE") == "replay"    # analyse the saved campaign: no mining, nothing overwritten
os.environ.setdefault("FA_CAMPAIGN", "vm_20261029" if REPLAY else "new_campaign")
import numpy as np, pandas as pd
import exp_runners as pr
import exp_data as pdd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
OUT = BUNDLE / "results" / "derived" / pr.CAMPAIGN if REPLAY else pr.RESULTS
OUT.mkdir(parents=True, exist_ok=True)
print("campaign", pr.CAMPAIGN, "->", pr.RESULTS)
print("build", pr.build_id(), "| extension", pr.ext_path())

In [ ]:
DATASETS = ["Chainstore", "Kosarak", "Instacart", "T20.I4.D1000k.N50k.L50k",
            "T30.I8.D2000k.N10k.L10k", "T25.I8.D5000k.N40k.L40k"]
S = 1e-4
K2 = ["hybcli_load", "hybcli_pairs_v0", "hybcli_pairs_v2", "hybcli_pairs_v0_1t", "hybcli_pairs_v2_1t",
      "borgelt_apriori", "borgelt_eclat", "borgelt_fpgrowth", "lcm"]
K3 = ["hybcli_load", "hybcli_anchor", "hybcli_seeded", "borgelt_eclat", "borgelt_fpgrowth", "lcm"]

In [ ]:
if not REPLAY:
    pr.run_grid("memory_boundary", DATASETS, K2, ks=[2], supports=[S], timeout_s=1500)
    pr.run_grid("memory_boundary", DATASETS, K3, ks=[3], supports=[S], timeout_s=1500)
s = pr.summarize(pr.load_results("memory_boundary"))
rss = s.pivot_table(index=["dataset", "k"], columns="method", values="peak_rss_kb") / 1024
base = rss.pop("hybcli_load") if "hybcli_load" in rss else 0
delta = rss.filter(like="hybcli_").sub(base, axis=0).add_suffix("_minus_load")
display(rss.round(0)); display(delta.round(0))

## Per-term memory model
index ≈ 2 · rows · 4 B (CSR both ways) + offsets; anchor TID cache at level k = Σ_{σ∈F_{k−1}} c(σ) · 4 B
(upper bound: Roaring compresses large lists); thread buffers = threads · N · 8 B; dense
triangular array (Borgelt Apriori) = |F1|²/2 · 4 B.

In [ ]:
if REPLAY:
    mt = pd.read_csv(pr.RESULTS / "memory_terms.csv")
else:
    from fastapriori import find_associations
    THREADS = os.cpu_count()
    terms = []
    for d in DATASETS:
        df = pdd.load_df(d)
        st = pdd.cost_stats(df, (S,))[0]
        f2 = find_associations(df, transaction_col="txn_id", item_col="item", k=2, min_support=S,
                               min_confidence=0.0, sorted_by=None, low_memory=False)
        anchor_cache_k3 = float(f2["instances"].sum() / 2) * 4     # each pair appears twice
        terms.append(dict(dataset=d, rows=st["rows"], N=st["N"], F1=st["F1"],
                          index_MB=(2 * st["rows"] * 4 + (st["D"] + st["N"]) * 4) / 2**20,
                          anchor_cache_k3_MB=anchor_cache_k3 / 2**20,
                          thread_buffers_MB=THREADS * st["N"] * 8 / 2**20,
                          triangular_MB=st["F1"] ** 2 / 2 * 4 / 2**20,
                          output_k2_MB=f2.memory_usage(deep=True).sum() / 2**20))
        del df, f2
    mt = pd.DataFrame(terms); mt.to_csv(pr.RESULTS / "memory_terms.csv", index=False)
mt.round(1)
